<a href="https://colab.research.google.com/github/sc22lg/my_pizza/blob/EG_version/my_eval_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pizza vs Clock Experiment (Transformer)

### Setup

In [1]:
# Import stuff
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
import tqdm

import random
import time

from pathlib import Path
import pickle
import os

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"
import plotly.graph_objects as go

from torch.utils.data import DataLoader

from functools import *
import pandas as pd
import gc

# import comet_ml
import itertools
# import comet_ml
import wandb
import itertools
class HookPoint(nn.Module):
    def __init__(self):
        super().__init__()
        self.fwd_hooks = []
        self.bwd_hooks = []
    def give_name(self, name):
        self.name = name
    def add_hook(self, hook, dir='fwd'):
        def full_hook(module, module_input, module_output):
            return hook(module_output, name=self.name)
        if dir=='fwd':
            handle = self.register_forward_hook(full_hook)
            self.fwd_hooks.append(handle)
        elif dir=='bwd':
            handle = self.register_backward_hook(full_hook)
            self.bwd_hooks.append(handle)
        else:
            raise ValueError(f"Invalid direction {dir}")
    def remove_hooks(self, dir='fwd'):
        if (dir=='fwd') or (dir=='both'):
            for hook in self.fwd_hooks:
                hook.remove()
            self.fwd_hooks = []
        if (dir=='bwd') or (dir=='both'):
            for hook in self.bwd_hooks:
                hook.remove()
            self.bwd_hooks = []
        if dir not in ['fwd', 'bwd', 'both']:
            raise ValueError(f"Invalid direction {dir}")
    def forward(self, x):
        return x

class Embed(nn.Module):
    def __init__(self, d_vocab, d_model):
        super().__init__()
        self.W_E = nn.Parameter(torch.randn(d_model, d_vocab)/np.sqrt(d_model))
    def forward(self, x):
        return torch.einsum('dbp -> bpd', self.W_E[:, x])

class Unembed(nn.Module):
    def __init__(self, d_vocab, d_model):
        super().__init__()
        self.W_U = nn.Parameter(torch.randn(d_model, d_vocab)/np.sqrt(d_vocab))
    def forward(self, x):
        return (x @ self.W_U)

# Positional Embeddings
class PosEmbed(nn.Module):
    def __init__(self, max_ctx, d_model):
        super().__init__()
        self.W_pos = nn.Parameter(torch.randn(max_ctx, d_model)/np.sqrt(d_model))
    def forward(self, x):
        return x+self.W_pos[:x.shape[-2]]

# Attention
class Attention(nn.Module):
    def __init__(self, d_model, num_heads, d_head, n_ctx, attn_coeff):
        super().__init__()
        self.W_K = nn.Parameter(torch.randn(num_heads, d_head, d_model)/np.sqrt(d_model))
        self.W_Q = nn.Parameter(torch.randn(num_heads, d_head, d_model)/np.sqrt(d_model))
        self.W_V = nn.Parameter(torch.randn(num_heads, d_head, d_model)/np.sqrt(d_model))
        self.W_O = nn.Parameter(torch.randn(d_model, d_head * num_heads)/np.sqrt(d_model))
        self.attn_coeff = attn_coeff
        self.register_buffer('mask', torch.tril(torch.ones((n_ctx, n_ctx))))
        self.d_head = d_head
        self.hook_k = HookPoint()
        self.hook_q = HookPoint()
        self.hook_v = HookPoint()
        self.hook_z = HookPoint()
        self.hook_attn = HookPoint()
        self.hook_attn_pre = HookPoint()

    def forward(self, x):
        k = self.hook_k(torch.einsum('ihd,bpd->biph', self.W_K, x))
        q = self.hook_q(torch.einsum('ihd,bpd->biph', self.W_Q, x))
        v = self.hook_v(torch.einsum('ihd,bpd->biph', self.W_V, x))
        attn_scores_pre = torch.einsum('biph,biqh->biqp', k, q)
        attn_scores_masked =attn_scores_pre
        attn_matrix = self.hook_attn(
            F.softmax(self.hook_attn_pre(attn_scores_masked/np.sqrt(self.d_head)), dim=-1)\
            *self.attn_coeff+(1-self.attn_coeff))
        z = self.hook_z(torch.einsum('biph,biqp->biqh', v, attn_matrix))
        z_flat = einops.rearrange(z, 'b i q h -> b q (i h)')
        out = torch.einsum('df,bqf->bqd', self.W_O, z_flat)
        return out

class MLP(nn.Module):
    def __init__(self, d_model, d_mlp, act_type):
        super().__init__()
        self.W_in = nn.Parameter(torch.randn(d_mlp, d_model)/np.sqrt(d_mlp))
        self.b_in = nn.Parameter(torch.zeros(d_mlp))
        self.W_out = nn.Parameter(torch.randn(d_model, d_mlp)/np.sqrt(d_model))
        self.b_out = nn.Parameter(torch.zeros(d_model))
        self.act_type = act_type
        # self.ln = LayerNorm(d_mlp, model=self.model)
        self.hook_pre = HookPoint()
        self.hook_post = HookPoint()
        assert act_type in ['ReLU', 'GeLU', 'Tanh']

    def forward(self, x):
        x = self.hook_pre(torch.einsum('md,bpd->bpm', self.W_in, x) + self.b_in)
        if self.act_type=='ReLU':
            x = F.relu(x)
        elif self.act_type=='GeLU':
            x = F.gelu(x)
        elif self.act_type=='Tanh':
            x = F.tanh(x)
        x = self.hook_post(x)
#        return x
        x = torch.einsum('dm,bpm->bpd', self.W_out, x) + self.b_out
        return x

# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_head, num_heads, n_ctx, act_type, attn_coeff):
        super().__init__()
        self.attn = Attention(d_model, num_heads, d_head, n_ctx, attn_coeff=attn_coeff)
        self.mlp = MLP(d_model, d_model*4,act_type)
        self.hook_attn_out = HookPoint()
        self.hook_mlp_out = HookPoint()
        self.hook_resid_pre = HookPoint()
        self.hook_resid_mid = HookPoint()
        self.hook_resid_post = HookPoint()

    def forward(self, x):
        x = self.hook_resid_mid(x + self.hook_attn_out(self.attn(self.hook_resid_pre(x))))
        x = self.hook_resid_post(x + self.hook_mlp_out(self.mlp(x)))
        return x

# Full transformer
class Transformer(nn.Module):
    def __init__(self, num_layers, d_vocab, d_model, d_head, num_heads, n_ctx, act_type, attn_coeff, use_cache=False, use_ln=True):
        super().__init__()
        assert 0<=attn_coeff<=1
        #print('parameters', num_layers, d_vocab, d_model, d_head, num_heads, n_ctx, act_type, attn_coeff, use_cache, use_ln)
        self.cache = {}
        self.use_cache = use_cache

        self.embed = Embed(d_vocab, d_model)
        self.pos_embed = PosEmbed(n_ctx, d_model)
        self.unembed = Unembed(d_vocab, d_model)
        self.use_ln = use_ln
        self.blocks = nn.ModuleList([TransformerBlock(d_model, d_head, num_heads, n_ctx, act_type, attn_coeff) for i in range(num_layers)])

        for name, module in self.named_modules():
            if type(module)==HookPoint:
                module.give_name(name)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos_embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.unembed(x)
        return x
    def forward_h(self, x):
        x = self.embed(x)
        tmp=x
        x = self.pos_embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.unembed(x)
        return tmp,x
    def forward_b(self, x):
        x = self.embed(x)
        tmp=x
        x = self.pos_embed(x)
        for blk in self.blocks:
            x = blk(x)
        return x

    def set_use_cache(self, use_cache):
        self.use_cache = use_cache

    def hook_points(self):
        return [module for name, module in self.named_modules() if 'hook' in name]

    def remove_all_hooks(self):
        for hp in self.hook_points():
            hp.remove_hooks('fwd')
            hp.remove_hooks('bwd')

    def cache_all(self, cache, incl_bwd=False):
        # Caches all activations wrapped in a HookPoint
        def save_hook(tensor, name):
            cache[name] = tensor.detach()
        def save_hook_back(tensor, name):
            cache[name+'_grad'] = tensor[0].detach()
        for hp in self.hook_points():
            hp.add_hook(save_hook, 'fwd')
            if incl_bwd:
                hp.add_hook(save_hook_back, 'bwd')

    def parameters_norm(self):
        # Returns the l2 norm of all parameters
        return sum([torch.sum(p*p).item() for p in self.parameters()])**0.5

    def l2_norm(self):
        # Returns the l2 norm of all parameters
        return sum([torch.sum(p*p) for p in self.parameters()])

    def parameters_flattened(self):
        # Returns all parameters as a single tensor
        return torch.cat([p.view(-1) for p in self.parameters()]).detach().cpu().numpy()
#DEVICE='cuda'
DEVICE='cpu'
print(f'Running on {DEVICE}')
class MyAddDataSet(torch.utils.data.Dataset):
    def __init__(self, func, C, diff_vocab=False, eqn_sign=False):
        self.func = func
        dim = 2
        self.dim = dim
        self.C = C
        self.inputs = []
        self.outputs = []
        self.vocab=C
        if diff_vocab:
            self.vocab*=2
        if eqn_sign:
            self.vocab+=1
            self.dim+=1
        self.vocab_out=0
        for p in range(C**dim):
            x = np.unravel_index(p, (C,)*dim)
            o=self.func(x)
            s=[x[0],x[1]]
            if diff_vocab:
                s[1]+=C
            if eqn_sign:
                s.append(self.vocab-1)
            self.inputs.append(s)
            self.outputs.append(o)
            self.vocab_out=max(self.vocab_out, o+1)
        if self.vocab_out!=C:
            print(f'warning {self.vocab_out=} neq to {C=}')
        self.inputs = torch.tensor(self.inputs, dtype=torch.long, device=DEVICE)
        self.outputs = torch.tensor(self.outputs, dtype=torch.long, device=DEVICE)
        #print(self.inputs,self.outputs)
    def __len__(self):
        return len(self.outputs)
    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

def cross_entropy_high_precision(logits, labels):
    # Shapes: batch x vocab, batch
    # Cast logits to float64 because log_softmax has a float32 underflow on overly
    # confident data and can only return multiples of 1.2e-7 (the smallest float x
    # such that 1+x is different from 1 in float32). This leads to loss spikes
    # and dodgy gradients
    logprobs = F.log_softmax(logits.to(torch.float64), dim=-1)
    prediction_logprobs = torch.gather(logprobs, index=labels[:, None], dim=-1)
    loss = -torch.mean(prediction_logprobs)
    return loss
def run_experiment(config, silent=False):
    exp_name=config['name']
    if not silent:
        print('parsing func',config['funcs'])
    config['func']=eval(config['funcs'])
    full_dataset = MyAddDataSet(func=config['func'],C=config['C'],diff_vocab=config['diff_vocab'],eqn_sign=config['eqn_sign'])
    model = Transformer(
        num_layers=config.get('n_layers',1),
        num_heads=config['n_heads'],
        d_model=config['d_model'],
        d_head=config.get('d_head',config['d_model']//config['n_heads']),
        attn_coeff=config['attn_coeff'],
        d_vocab=full_dataset.vocab,
#        attention_dir=config.get('attention_dir','bidirectional'),
        act_type=config.get('act_fn','relu'),
        n_ctx=full_dataset.dim,
#        normalization_type=None,
    )
    model.to(DEVICE)
    train_size = int(config['frac'] * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])
    if not silent:
        print('random split',len(train_dataset),len(test_dataset))
    batch_size = config.get('batch_size',len(full_dataset))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    opt = optim.AdamW(model.parameters(),lr=config.get('lr',1e-3),weight_decay=config.get('weight_decay',1e-4),betas=(0.9,0.98))
    scheduler = optim.lr_scheduler.LambdaLR(opt, lambda step: min(step/10, 1)) # 10 epoch warmup
    if not silent:
        print(config.get('lr',1e-3),config.get('weight_decay',1e-4))
        print(opt,scheduler)
    losses=[]
    accs=[]
    losses_val=[]
    accs_val=[]
    norms=[]
    loss_val=10
    acc_val=0
    stop=None
    best_train_acc=0.
    best_test_acc=0.
    perfect_train_time=None
    perfect_test_time=None
    pbar = range(config.get('epoch',10000))
    if not silent:
        pbar=tqdm.tqdm(pbar)
    gaps=[]
    early_stop_a=2
    early_stop_b=1
    if config.get('early_stop',None) is not None:
        early_stop_a, early_stop_b = config['early_stop']
    early_stop_timer=0
    #model.train()
    run = None#wandb.init(reinit=True,config=config,project='modadd_new')#,settings=wandb.Settings(start_method="spawn"))
    try:
        for i in pbar:
            def evaluation():
                nonlocal best_test_acc
                nonlocal perfect_test_time
                nonlocal early_stop_timer
                nonlocal early_stop_a
                nonlocal early_stop_b
                # evaluate on test set, return loss and accuracy
                # with torch.inference_mode():
                    #model.eval()
                losses_eval=[]
                accs_eval=[]
                for inp,ans in test_loader:
                    # print(inp.shape)
                    out = model(inp)[:,-1,:]
                    loss = cross_entropy_high_precision(out,ans)
                    acc = torch.sum((out.argmax(dim=1)==ans).float())/len(ans)
                    # print(inp,'test',out.argmax(dim=1),ans)
#                    acc = (out.argmax(dim=1)==ans).float().mean()
                    losses_eval.append(loss.item())
                    accs_eval.append(acc.item())
                    # print(loss,acc)
                #print(losses_eval,accs_eval)
                eval_loss, eval_acc = np.mean(losses_eval), np.mean(accs_eval)
                best_test_acc = max(best_test_acc, eval_acc)
                if eval_acc==1. and perfect_test_time is None:
                    perfect_test_time = i
                if eval_acc>=early_stop_a:
                    early_stop_timer+=1
                else:
                    early_stop_timer=0
                #print(eval_loss,eval_acc)
                return eval_loss, eval_acc
            if early_stop_timer>=early_stop_b:
                break
            for inp,ans in train_loader:
                #print(inp.shape,inp.dtype)
                # print(inp,'train')
                #print(len(inp))
                #model.train()
                out = model(inp)[:,-1,:]
                loss = cross_entropy_high_precision(out,ans)
                loss_val, acc_val = evaluation()
                #print(loss_val,acc_val)
                loss.backward()
                # clip gradients
                #if config.get('clip',None) is not None:
                #    nn.utils.clip_grad_norm_(model.parameters(), config['clip'])
                opt.step()
                scheduler.step()
                opt.zero_grad()
                acc = (out.argmax(dim=1)==ans).float().mean()
                norm = sum([torch.sum(p*p).item() for p in model.parameters()])**0.5
                #sum(p.norm()**2 for p in model.parameters()).sqrt().item()
                losses.append(loss.item())
                accs.append(acc.item())
                losses_val.append(loss_val)
                accs_val.append(acc_val)
                norms.append(norm)

                best_train_acc=max(best_train_acc,acc.item())
                if acc.item()==1. and perfect_train_time is None:
                    perfect_train_time = i
                gaps.append(best_train_acc-best_test_acc)
                if pbar is tqdm.tqdm:
                    pbar.set_description(f"loss: {loss.item():.3f}, accm: {best_train_acc:.3f}, vloss: {loss_val:.3f}, vaccm: {best_test_acc:.3f}, norm: {norm:.3f}, acc: {acc.item():.3f}, vacc: {acc_val:.3f}")
                #print(f"loss: {loss.item():.3f}, accm: {best_train_acc:.3f}, vloss: {loss_val:.3f}, vaccm: {best_test_acc:.3f}, norm: {norm:.3f}, acc: {acc.item():.3f}, vacc: {acc_val:.3f}")
                if run:
                    run.log({'training_loss': loss.item(),
                    'validation_loss': loss_val,
                    'training_accuracy': acc.item(),
                    'validation_accuracy': acc_val,
                    'parameter_norm': norm,
                    'best_train_accuracy': best_train_acc,
                    'best_test_accuracy': best_test_acc,
                    'generalization_gap': best_train_acc-best_test_acc,
                    'generalization_delay1': sum(gaps)})
    except KeyboardInterrupt:
        print('Keyboard interrupt. Gracefully exiting...')
        pass
    if not silent:
        print('Finished.')
    generalization_gap=best_train_acc-best_test_acc
    generalization_delay1=sum(gaps)
    generalization_delay2=sum(max(t-(best_train_acc-best_test_acc),0) for t in gaps)
    if run:
        run.summary["generalization_delay2"] = generalization_delay2
    # run.finish()
    return dict(
        losses=losses,
        accs=accs,
        losses_val=losses_val,
        accs_val=accs_val,
        norms=norms,
        model=model,
        config=config,
        generalization_gap=generalization_gap,
        generalization_delay1=generalization_delay1,
        generalization_delay2=generalization_delay2,
        best_train_acc=best_train_acc,
        best_test_acc=best_test_acc,
        perfect_train_time=perfect_train_time,
        perfect_test_time=perfect_test_time,
        dataset = full_dataset,
        run=run
    )

Running on cpu


## Model Metrics Test




In [31]:
import json
from sklearn.decomposition import PCA
import math
import seaborn as sns
!pip install torch_geometric
from torch_geometric.nn import functional

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.3 MB/s eta 0:00:00


In [3]:
# l2-norm of model?
def l2_norm(model, model_file):
  print(f"l2-norm of model: {model.parameters_norm():.3f}")

### Gini Coefficient

In [63]:
def gini_coefficient(model, model_file):
  gini_values = []
  for name, param in model.named_parameters():
    # Consider only weight matrices (typically dim > 1)
    if param.dim() > 1:
      detached_param = param.cpu().detach()
      if detached_param.dim() == 2:
        # For 2D tensors, functional.gini is applied directly
        gini_val = functional.gini(detached_param)
        gini_values.append(gini_val)
      elif detached_param.dim() == 3:
        # For 3D tensors, functional.gini expects 2D input.
        # We iterate over the first dimension (e.g., heads for attention weights)
        # and calculate gini for each 2D slice, then average these values.
        gini_per_slice = [functional.gini(detached_param[i]) for i in range(detached_param.shape[0])]
        if gini_per_slice:
            gini_values.append(np.mean(gini_per_slice))
        else:
            gini_values.append(0.0)
      # For higher dimensions (e.g., 4D), additional specific handling would be needed.
      # Based on the model architecture, weight matrices are typically 2D or 3D.

  if gini_values:
    return np.mean(gini_values)
  else:
    return 0.0 # Return 0 if no weight matrices are found

### Distance Irrelevance

In [8]:
def distance_irrelevance(model, model_file):
  model=result_modadd['model']
  model.load_state_dict(torch.load(model_file,map_location=DEVICE))
  dataset = result_modadd['dataset']
  dataloader = torch.utils.data.DataLoader(dataset, batch_size=C*C)
  model = result_modadd['model']
  oo=[[0]*C for _ in range(C)]
  for x,y in dataloader:
      #print(x)
      with torch.inference_mode():
          model.eval()
          model.remove_all_hooks()
          o=model(x)[:,-1,:]
          o0=o[list(range(len(x))),y]
          o0=o0.cpu()
          x=x.cpu()
          for p,q in zip(o0,x):
              A,B=int(q[0].item()),int(q[1].item())
              oo[(A+B)%C][(A-B)%C]=p.item()
  oo=np.array(oo)

  dd=np.mean(np.std(oo,axis=0))/np.std(oo.flatten())
  '''
  sns.heatmap(np.array(oo))
  plt.xlabel(f'(a-b) mod {C}')
  plt.ylabel(f'(a+b) mod {C}')
  plt.title('Correct Logits')
  '''
  return dd

### Gradient Symmetricity

In [9]:
def gradient_symmetricity(model, model_file):
  model.load_state_dict(torch.load(model_file,map_location='cpu'))
  # we=model.embed.W_E.T
  # from sklearn.decomposition import PCA
  # pca = PCA(n_components=20)
  # we2=pca.fit_transform(we.detach().cpu().numpy())
  oc=None
  tt=0
  xs=[(a,b,c) for a in range(C) for b in range(C) for c in range(C)]
  random.Random(42).shuffle(xs)
  xs=xs[:100]
  for abc in xs:
      a,b,c=abc
      x=torch.tensor([[a,b]],device='cpu')
      t,o0=model.forward_h(x)
      model.zero_grad()
      #print(a,b,c)
      model.remove_all_hooks()
      o=o0[0,-1,:]
      t.retain_grad()
      o[c].backward(retain_graph=True)
      tg=t.grad[0].detach().cpu().numpy()
      #tt+=tg[0][0]
      dp=np.sum(tg[0]*tg[1])/np.sqrt(np.sum(tg[0]**2))/np.sqrt(np.sum(tg[1]**2))
      tt+=dp
  return tt/len(xs)

### Circularity

In [10]:
def circularity(model, model_file):
  model.load_state_dict(torch.load(model_file,map_location=DEVICE))
  we=model.embed.W_E.T
  pca = PCA(n_components=20)
  we2=pca.fit_transform(we.detach().cpu().numpy())
  def ang(x):
      return math.cos(x)+math.sin(x)*1j
  rst=0
  first_k=4
  for ix in range(first_k):
      vs=we2[:,ix]*1
      vs=vs/np.sqrt(np.sum(vs*vs))/math.sqrt(59)
      tt=[]
      for i in range(1,59):
          vv=[vs[t*i%59] for t in range(59)]
          sa=sum(vv[t]*ang(2*math.pi*t/59) for t in range(59))
          tt.append((-abs(sa)**2*2,i))
      tt.sort()
      i=tt[0][1]
      rst+=max(min(-tt[0][0],1.),0.)
      i=min(i,59-i)
      v=[vs[t*i%59] for t in range(59)]
  rst/=first_k
  return rst

### Load Trained Model

In [64]:
import pandas as pd
# Here you can specify the model to load
model_type = 'EG'
num_models = 100

all_model_metrics = []

for i in range(1, num_models+1):
  runid = f"{model_type}_{i}"

  with open(f'save/config_{runid}.json','r') as f:
      config=json.load(f)
  model_file=f'save/model_{runid}.pt'
  C=59
  #print(config)
  config.setdefault('diff_vocab',False)
  config.setdefault('eqn_sign',False)
  config['epoch']=2
  result_modadd=run_experiment(config, True)
  model=result_modadd['model']
  dataset = result_modadd['dataset']
  dataloader = torch.utils.data.DataLoader(dataset, batch_size=C*C)
  model.load_state_dict(torch.load(model_file,map_location=DEVICE))

  circ = circularity(model, model_file)
  print(f"Circularity: {circ:.3f}")
  grad_sym = gradient_symmetricity(model, model_file)
  print(f"Gradient Symmetricity: {grad_sym:.3f}")
  dist_irr = distance_irrelevance(model, model_file)
  print(f"Distance Irrelevance: {dist_irr:.3f}")
  gini_val = gini_coefficient(model, model_file)

  all_model_metrics.append({
      'runid': runid,
      'Circularity': circ,
      'Gradient Symmetricity': grad_sym,
      'Distance Irrelevance': dist_irr,
      'Gini Coefficient': gini_val
  })

metrics_df = pd.DataFrame(all_model_metrics)
metrics_df.to_csv(f'{model_type}_model_metrics.csv', index=False)
print("All model metrics saved to model_metrics.csv")

Circularity: 0.253
Gradient Symmetricity: 0.999
Distance Irrelevance: 0.754
Circularity: 0.423
Gradient Symmetricity: 0.991
Distance Irrelevance: 0.684
Circularity: 0.385
Gradient Symmetricity: 0.921
Distance Irrelevance: 0.822
Circularity: 0.294
Gradient Symmetricity: 0.952
Distance Irrelevance: 0.759
Circularity: 0.226
Gradient Symmetricity: 1.000
Distance Irrelevance: 0.625
Circularity: 0.265
Gradient Symmetricity: 0.967
Distance Irrelevance: 0.719
Circularity: 0.331
Gradient Symmetricity: 0.945
Distance Irrelevance: 0.869
Circularity: 0.433
Gradient Symmetricity: 0.759
Distance Irrelevance: 0.863
Circularity: 0.301
Gradient Symmetricity: 0.974
Distance Irrelevance: 0.689
Circularity: 0.313
Gradient Symmetricity: 0.941
Distance Irrelevance: 0.683
Circularity: 0.330
Gradient Symmetricity: 0.969
Distance Irrelevance: 0.783
Circularity: 0.229
Gradient Symmetricity: 0.999
Distance Irrelevance: 0.474
Circularity: 0.341
Gradient Symmetricity: 0.959
Distance Irrelevance: 0.695
Circularity:

In [16]:
for p in model.named_parameters():
  print(p)

('embed.W_E', Parameter containing:
tensor([[ 1.2383e-06,  8.2953e-05, -9.1122e-06,  ...,  3.8723e-06,
          5.6158e-06, -4.0192e-05],
        [-2.3096e-05, -1.3047e-02,  1.2273e-04,  ...,  2.0404e-07,
          1.1969e-06,  2.4532e-01],
        [-4.1224e-11, -4.4521e-07, -1.3576e-03,  ..., -8.2512e-09,
          7.6509e-07,  1.0450e-02],
        ...,
        [-2.4916e-09, -5.6649e-04,  8.2503e-03,  ...,  4.6002e-01,
          2.9138e-08,  3.1240e-03],
        [-1.1431e-07,  3.1589e-05,  1.7702e-04,  ...,  2.6804e-05,
         -1.9119e-05,  2.2439e-08],
        [ 1.2395e-04, -1.2158e-07,  2.4035e-07,  ..., -2.2953e-02,
          3.0188e-03, -3.0053e-05]], requires_grad=True))
('pos_embed.W_pos', Parameter containing:
tensor([[ 1.4467e-06,  7.4534e-11,  8.8271e-04, -8.3634e-04, -4.1180e-08,
         -2.8704e-05, -4.9495e-03,  1.0521e-08, -7.9763e-02, -1.0631e-06,
         -1.1066e-06, -2.3798e-06,  8.0734e-07, -2.4111e-06, -2.1863e-08,
          1.2030e-06, -1.2453e-06,  1.1287e-04,

### Read in data and plot EG and GD metrics

In [52]:
import pandas as pd
import plotly.express as px

# Read the EG_model_metrics.csv file and add a model_type column
eg_metrics_df = pd.read_csv('EG_model_metrics.csv')
eg_metrics_df['model_type'] = 'EG'

# Read the GD_model_metrics.csv file and add a model_type column
gd_metrics_df = pd.read_csv('GD_model_metrics.csv')
gd_metrics_df['model_type'] = 'GD'

# Concatenate the two dataframes
metrics_df = pd.concat([eg_metrics_df, gd_metrics_df], ignore_index=True)

# Create a new column for marker shape based on Circularity
metrics_df['Circularity_Group'] = metrics_df['Circularity'].apply(lambda x: 'Circularity < 0.95' if x < 0.95 else 'Circularity >= 0.95')

# Generate the scatter plot
fig = px.scatter(
    metrics_df,
    x='Distance Irrelevance',
    y='Gradient Symmetricity',
    color='model_type', # Color by model type
    color_discrete_map={'EG': 'purple', 'GD': 'orange'}, # Specify colors
    symbol='Circularity_Group', # Use symbol to distinguish shapes
    symbol_map={'Circularity < 0.95': 'square', 'Circularity >= 0.95': 'circle'},
    hover_name='runid',
    title='Gradient Symmetricity vs. Distance Irrelevance by Circularity and Model Type',
    labels={
        'Distance Irrelevance': 'Distance Irrelevance',
        'Gradient Symmetricity': 'Gradient Symmetricity',
        'Circularity_Group': 'Circularity',
        'model_type': 'Model Type'
    }
)

# Set axis ranges from 0 to 1
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])

# Significantly increase the font size of the plot elements
fig.update_layout(
    title=None, # Removed the title
    title_font_size=24,
    xaxis=dict(
        title_font_size=20,
        tickfont_size=16
    ),
    yaxis=dict(
        title_font_size=20,
        tickfont_size=16
    ),
    legend=dict(
        font_size=18,
        x=0.01,
        y=0.01,
        xanchor='left',
        yanchor='bottom'
    )
)

fig.show()

In [65]:
import pandas as pd

# Calculate average metrics for EG models
eg_avg_metrics = eg_metrics_df[['Circularity', 'Gradient Symmetricity', 'Distance Irrelevance', 'Gini Coefficient']].mean()

# Calculate average metrics for GD models
gd_avg_metrics = gd_metrics_df[['Circularity', 'Gradient Symmetricity', 'Distance Irrelevance', 'Gini Coefficient']].mean()

# Create a dictionary to hold the average metrics
avg_metrics_data = {
    'EG': eg_avg_metrics,
    'GD': gd_avg_metrics
}

# Create a DataFrame from the dictionary, transposing it to have model types as rows
avg_metrics_table = pd.DataFrame(avg_metrics_data).T

# Rename columns for better readability if needed (they should already be correct)
avg_metrics_table.columns = ['Circularity', 'Gradient Symmetricity', 'Distance Irrelevance', 'Gini Coefficient']

print("Average Metrics Table:")
print(avg_metrics_table)

Average Metrics Table:
    Circularity  Gradient Symmetricity  Distance Irrelevance  Gini Coefficient
EG          0.0                    0.0                   0.0          0.921702
GD          0.0                    0.0                   0.0          0.707438


### Model algorithm classification

In [ ]:
def classify_model(row):
    circularity = row['Circularity']
    gs = row['Gradient Symmetricity']
    di = row['Distance Irrelevance']

    if circularity >= 0.95 and gs <= 0.99 and di >= 0.4:
        return 'Clock'
    elif circularity >= 0.95 and gs >= 0.95 and di <= 0.4:
        return 'Pizza'
    elif circularity <= 0.95:
        return 'Non-circular'
    else:
        return 'Unknown'

metrics_df['Model_Class'] = metrics_df.apply(classify_model, axis=1)

# Define color map for model classes
color_map = {
    'Clock': 'blue',
    'Pizza': 'green',
    'Non-circular': 'red'
}

# Filter for EG models and plot
eg_df = metrics_df[metrics_df['model_type'] == 'EG']
fig_eg = px.scatter(
    eg_df,
    x='Distance Irrelevance',
    y='Gradient Symmetricity',
    color='Model_Class',
    color_discrete_map=color_map,
    hover_name='runid',
    title='',
    labels={
        'Distance Irrelevance': 'Distance Irrelevance',
        'Gradient Symmetricity': 'Gradient Symmetricity',
        'Model_Class': 'Model Class'
    }
)
fig_eg.update_xaxes(range=[0, 1])
fig_eg.update_yaxes(range=[0, 1])
fig_eg.update_layout(
    title_font_size=24,
    xaxis=dict(title_font_size=20, tickfont_size=16),
    yaxis=dict(title_font_size=20, tickfont_size=16),
    legend=dict(font_size=18, x=0.01, y=0.01, xanchor='left', yanchor='bottom')
)
fig_eg.show()

# Filter for GD models and plot
gd_df = metrics_df[metrics_df['model_type'] == 'GD']
fig_gd = px.scatter(
    gd_df,
    x='Distance Irrelevance',
    y='Gradient Symmetricity',
    color='Model_Class',
    color_discrete_map=color_map,
    hover_name='runid',
    title='',
    labels={
        'Distance Irrelevance': 'Distance Irrelevance',
        'Gradient Symmetricity': 'Gradient Symmetricity',
        'Model_Class': 'Model Class'
    }
)
fig_gd.update_xaxes(range=[0, 1])
fig_gd.update_yaxes(range=[0, 1])
fig_gd.update_layout(
    title_font_size=24,
    xaxis=dict(title_font_size=20, tickfont_size=16),
    yaxis=dict(title_font_size=20, tickfont_size=16),
    legend=dict(font_size=18, x=0.01, y=0.01, xanchor='left', yanchor='bottom')
)
fig_gd.show()

### Circularity plot


In [ ]:
non_circular_models = metrics_df[(metrics_df['Model_Class'] == 'Non-circular') & (metrics_df['model_type'].isin(['EG', 'GD']))]

mean_circularity_eg = non_circular_models[non_circular_models['model_type'] == 'EG']['Circularity'].mean()
mean_circularity_gd = non_circular_models[non_circular_models['model_type'] == 'GD']['Circularity'].mean()
print(f"\nMean Circularity of Non-circular EG models: {mean_circularity_eg:.3f}")
print(f"Mean Circularity of Non-circular GD models: {mean_circularity_gd:.3f}")

# Create a box and whisker plot of circularity for the non-circular EG and GD models
fig = px.box(
    non_circular_models,
    x='model_type',
    y='Circularity',
    color='model_type',
    color_discrete_map={'EG': 'purple', 'GD': 'orange'},
    title='',
    labels={
        'model_type': 'Model Type',
        'Circularity': 'Circularity'
    }
)

fig.update_layout(
    title_font_size=24,
    xaxis=dict(title_font_size=20, tickfont_size=16),
    yaxis=dict(title_font_size=20, tickfont_size=16)
)

fig.show()


Mean Circularity of Non-circular EG models: 0.342
Mean Circularity of Non-circular GD models: 0.868


### Gini plot

In [69]:
import pandas as pd
import plotly.express as px

# Create the bar chart using the pre-calculated average metrics
fig = px.bar(
    avg_metrics_table.reset_index(names=['model_type']), # Use the DataFrame with pre-calculated averages and reset index to create 'model_type' column
    x='model_type',
    y='Gini Coefficient',
    color='model_type', # Color by model type
    color_discrete_map={'EG': 'purple', 'GD': 'orange'}, # Specify colors
    labels={
        'Gini Coefficient': 'Average Gini Coefficient', # Updated label for clarity
        'model_type': 'Model Type'
    }
)

# Set y-axis range from 0 to 1
fig.update_yaxes(range=[0, 1])

# Significantly increase the font size of the plot elements
fig.update_layout(
    title_font_size=24,
    xaxis={
        'title_font_size': 20,
        'tickfont_size': 16,
        'tickangle': -45 # Rotate x-axis labels for better readability
    },
    yaxis={
        'title_font_size': 20,
        'tickfont_size': 16
    },
)

fig.show()